In [1]:
!pip install -q langgraph langchain-google-genai

import os
from typing import TypedDict, List
from google.colab import userdata
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.2 MB/s eta 0:00:00


In [2]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

coder_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_retries=1
)

class AutoSolverState(TypedDict):
    challenge_text: str
    algorithmic_blueprint: str
    cpp_source_code: str
    virtual_judge_review: str
    retry_tracker: int
    failure_memory: List[str]
    validation_cases: str

In [3]:
def formulate_strategy(state: AutoSolverState):
    prompt = f"""
    You are an expert competitive programmer.
    Analyze this Codeforces problem:
    {state['challenge_text']}

    Provide a detailed breakdown:
    1. Core Logic
    2. Algorithm
    3. Complexity
    4. Edge Cases
    Keep it concise and algorithmic.
    """
    response = coder_llm.invoke(prompt)
    state["algorithmic_blueprint"] = response.content
    return state

def draft_cpp_solution(state: AutoSolverState):
    past_errors = "\\n".join(state["failure_memory"])

    prompt = f"""
    Write a complete C++17 solution for the following problem.
    Problem: {state['challenge_text']}
    Blueprint: {state['algorithmic_blueprint']}
    Past Mistakes to Avoid: {past_errors}

    Output ONLY raw C++ code. No markdown formatting.
    """
    response = coder_llm.invoke(prompt)

    raw_code = response.content.strip()
    if raw_code.startswith("```cpp"):
        raw_code = raw_code[6:]
    if raw_code.endswith("```"):
        raw_code = raw_code[:-3]

    state["cpp_source_code"] = raw_code.strip()
    return state

def construct_test_suite(state: AutoSolverState):
    prompt = f"""
    Problem: {state['challenge_text']}
    Generate 3 rigorous test cases to validate a C++ solution.

    Format:
    1.
    Input:
    ...
    Expected Output:
    ...
    """
    response = coder_llm.invoke(prompt)
    state["validation_cases"] = response.content
    return state

In [4]:
def evaluate_submission(state: AutoSolverState):
    prompt = f"""
    You are a strict Codeforces Judge.
    Problem: {state['challenge_text']}
    Code: {state['cpp_source_code']}
    Tests: {state['validation_cases']}

    Check for logic, edge cases, and limits.
    If perfect, output exactly: ACCEPTED
    If failed, output exactly: REJECTED: [Reason]
    """
    response = coder_llm.invoke(prompt)
    state["virtual_judge_review"] = response.content.strip()
    return state

def record_failure_and_retry(state: AutoSolverState):
    state["failure_memory"].append(state["virtual_judge_review"])
    state["retry_tracker"] += 1
    return state

def route_workflow(state: AutoSolverState):
    verdict = state.get("virtual_judge_review", "").upper()

    if "ACCEPTED" in verdict:
        return END

    if state["retry_tracker"] >= 3:
        return END

    return "record_failure_and_retry"

In [5]:
workflow = StateGraph(AutoSolverState)

workflow.add_node("plan_algorithm", formulate_strategy)
workflow.add_node("write_code", draft_cpp_solution)
workflow.add_node("generate_tests", construct_test_suite)
workflow.add_node("run_virtual_judge", evaluate_submission)
workflow.add_node("record_failure_and_retry", record_failure_and_retry)

workflow.set_entry_point("plan_algorithm")

workflow.add_edge("plan_algorithm", "write_code")
workflow.add_edge("write_code", "generate_tests")
workflow.add_edge("generate_tests", "run_virtual_judge")

workflow.add_conditional_edges(
    "run_virtual_judge",
    route_workflow
)

workflow.add_edge("record_failure_and_retry", "write_code")

ai_competitive_programmer = workflow.compile()

In [6]:
target_challenge =  """G. Stripe, Token and Two Players

time limit per test: 3 seconds
memory limit per test: 256 megabytes

There is a stripe of n + 1 cells, numbered from 1 to n + 1.
Initially, there is a token with power 1 on cell number 1,
and the numbers a_1, a_2, ..., a_n are written on cells
1, 2, ..., n respectively.

Two players play a game. On each move, the player performs the following actions in order:

1. Suppose the token is on cell number i.
2. The player may increase the power of the token by any integer from 0 to a_i inclusive.
3. Then the player moves the token forward by any positive integer not exceeding the token's power, such that after this action the token does not leave the stripe.

The player after whose move the token lands on cell n + 1 wins.

Input

Each test contains multiple test cases.

The first line contains the number of test cases t
(1 ≤ t ≤ 10^4).

The description of the test cases follows.

The first line of each test case contains one integer n
(1 ≤ n ≤ 10^5) — the number of written numbers.

The second line of each test case contains n integers
a_1, a_2, ..., a_n
(0 ≤ a_i ≤ 10^9) — the numbers written on the cells.

It is guaranteed that the sum of n over all test cases does not exceed 10^5.

Output

For each test case, output one integer, 1 or 2 —
the number of the player who wins with optimal play.
(Player 1 makes the first move.)

Example

Input
4
3
0 0 0
3
1 1 2
5
0 0 1 0 0
9
0 1 2 0 0 1 0 0 0

Output
1
2
1
2

Note

In the first test case, the power of the token remains equal to 1 throughout the game,
so on every move the players must move exactly one cell forward.
Thus, 3 moves will be made, and the last move will be made by player 1,
so he will win in any case.

In the second test case, player 2 has a winning strategy:
on their very first move, increase the token's power as much as possible,
and then jump to cell 4 and win.

This is always possible, since at the start of the move the token will be on cell 2 or 3,
and its power can be increased to at least 2.
"""


initial_graph_state = {
    "challenge_text": target_challenge,
    "algorithmic_blueprint": "",
    "cpp_source_code": "",
    "virtual_judge_review": "",
    "retry_tracker": 0,
    "failure_memory": [],
    "validation_cases": ""
}


import time

print("Deploying Autonomous AI Solver...\n")

max_attempts = 5
for attempt in range(max_attempts):
    try:
        # Try to run the agent
        final_result = ai_competitive_programmer.invoke(initial_graph_state)
        break # If it succeeds, break out of the loop!

    except Exception as e:
        # If the server is busy, wait and try again
        if "503" in str(e):
            print(f"Server overloaded (Attempt {attempt + 1}/{max_attempts}). Waiting 10 seconds before retrying...")
            time.sleep(10)
        else:
            # If it's a different error, raise it normally
            raise e
else:
    print("\nFailed to connect to the model after 5 attempts. The servers are heavily congested. Take a quick break and try again in 10 minutes.")
print("="*60)
print("  AGENT EXECUTION REPORT")
print("="*60)

print("\n[3] HEALING METRICS")
print("-" * 25)
print(f"Total Retries Executed: {final_result['retry_tracker']}")

print("\n[4] FINAL C++ SOURCE CODE")
print("-" * 25)
print(final_result["cpp_source_code"])
print("\n" + "="*60)

Deploying Autonomous AI Solver...

Server overloaded (Attempt 1/5). Waiting 10 seconds before retrying...


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 34.240716472s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT